In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%bash
pip install -q mne

In [ ]:
import os
from pathlib import Path

def find_mumtaz_path():
    default_path = Path("/content/drive/MyDrive/Data/MUMTAZ")
    if default_path.exists():
        return default_path

    drive_root = Path("/content/drive/MyDrive")
    for p in drive_root.rglob("MUMTAZ"):
        if p.is_dir():
            return p
    return default_path

MUMTAZ_DIR = find_mumtaz_path()
OUTPUT_DIR = Path("/content/drive/MyDrive/J26-DS-348/outputs/c2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert MUMTAZ_DIR.exists(), f"MUMTAZ folder not found. Please place it in your Google Drive!"
edf_files = list(MUMTAZ_DIR.glob("*.edf"))
print(f"Using MUMTAZ folder at: {MUMTAZ_DIR}")
print(f"Found {len(edf_files)} EDF files")

Using MUMTAZ folder at: /content/drive/MyDrive/Data/MUMTAZ
Found 181 EDF files


In [ ]:
import math
import re
import time
import json
from dataclasses import dataclass
from typing import Tuple, Dict, Any, List

import mne
import numpy as np
from scipy.signal import butter, sosfiltfilt
import torch
import torch.nn as nn
from torch import Tensor
from torch.autograd import Function
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt

# Set device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training Device:", DEVICE)

# 19 standard 10-20 scalp channels
MUMTAZ_CHANNELS = (
    "Fp1", "F3", "C3", "P3", "O1",
    "F7", "T3", "T5", "Fz", "Fp2",
    "F4", "C4", "P4", "O2", "F8",
    "T4", "T6", "Cz", "Pz",
)

# PREPROCESSING & ADAPTIVE TBR ENCODER
def bandpass_filter(eeg: np.ndarray, fs: float, low: float = 0.5, high: float = 45.0, order: int = 4) -> np.ndarray:
    sos = butter(order, [low, high], btype="bandpass", fs=fs, output="sos")
    return sosfiltfilt(sos, eeg, axis=-1)

def make_windows(eeg: np.ndarray, fs: float, window_sec: float = 2.0) -> np.ndarray:
    win = int(round(window_sec * fs))
    n_samples = eeg.shape[-1]
    if n_samples < win:
        return np.empty((0, eeg.shape[0], win), dtype=eeg.dtype)
    starts = range(0, n_samples - win + 1, win)
    return np.stack([eeg[:, s : s + win] for s in starts])

def tbr_encode_dual_binary(windows: np.ndarray, thresholds: np.ndarray) -> torch.Tensor:
    # windows: [N, C, T], thresholds: [C]
    diffs = windows[:, :, 1:] - windows[:, :, :-1]
    th = thresholds[None, :, None]
    up = (diffs > th).astype(np.float32)
    down = (diffs < -th).astype(np.float32)
    # Pad silent first time-step
    up = np.pad(up, ((0, 0), (0, 0), (1, 0)))
    down = np.pad(down, ((0, 0), (0, 0), (1, 0)))
    # Transpose to [N, T, C] and cat to [N, T, 2C]
    up = np.transpose(up, (0, 2, 1))
    down = np.transpose(down, (0, 2, 1))
    dual_binary = np.concatenate((up, down), axis=-1)
    return torch.from_numpy(dual_binary).float()

# NEURON MODELS & PLASTICITY
class SuperSpikeSurrogate(Function):
    @staticmethod
    def forward(ctx, u: Tensor, gamma: float = 10.0) -> Tensor:
        ctx.save_for_backward(u)
        ctx.gamma = gamma
        return (u >= 0.0).to(dtype=u.dtype)

    @staticmethod
    def backward(ctx, grad_output: Tensor) -> Tuple[Tensor, None]:
        (u,) = ctx.saved_tensors
        grad_u = grad_output / (1.0 + ctx.gamma * torch.abs(u)).pow(2)
        return grad_u, None

def superspike(u: Tensor, gamma: float = 10.0) -> Tensor:
    return SuperSpikeSurrogate.apply(u, gamma)

class LIFNeuron(nn.Module):
    def __init__(self, size: int, beta_init: float = 0.9, v_th_init: float = 1.0, gamma: float = 10.0):
        super().__init__()
        self.size = size
        self.gamma = gamma
        self.beta = nn.Parameter(torch.full((size,), float(beta_init)))
        self.v_th = nn.Parameter(torch.full((size,), float(v_th_init)))

    def init_state(self, batch_size: int, device: torch.device) -> Tensor:
        return torch.zeros(batch_size, self.size, device=device)

    def step(self, current: Tensor, u_prev: Tensor) -> Tuple[Tensor, Tensor]:
        u_decayed = self.beta * u_prev + current
        spikes = superspike(u_decayed - self.v_th, self.gamma)
        u_next = u_decayed - spikes * self.v_th
        return spikes, u_next

    def clamp_parameters(self, v_th_min: float = 0.1):
        with torch.no_grad():
            self.beta.clamp_(0.01, 0.999)
            self.v_th.clamp_(min=v_th_min)

class ALIFNeuron(nn.Module):
    def __init__(self, size: int, beta_init: float = 0.9, v_th_init: float = 1.0, tau_ad_ms: float = 500.0, dt_ms: float = 4.0, beta_ad_init: float = 0.5, gamma: float = 10.0):
        super().__init__()
        self.size = size
        self.gamma = gamma
        rho_val = math.exp(-float(dt_ms) / float(tau_ad_ms))
        self.register_buffer("rho", torch.full((size,), rho_val))
        self.beta = nn.Parameter(torch.full((size,), float(beta_init)))
        self.v_th = nn.Parameter(torch.full((size,), float(v_th_init)))
        self.beta_ad = nn.Parameter(torch.full((size,), float(beta_ad_init)))

    def init_state(self, batch_size: int, device: torch.device) -> Tuple[Tensor, Tensor]:
        return torch.zeros(batch_size, self.size, device=device), torch.zeros(batch_size, self.size, device=device)

    def step(self, current: Tensor, state: Tuple[Tensor, Tensor]) -> Tuple[Tensor, Tuple[Tensor, Tensor], Tensor]:
        u_prev, a_prev = state
        A_eff = self.v_th + self.beta_ad * a_prev
        u_decayed = self.beta * u_prev + current
        spikes = superspike(u_decayed - A_eff, self.gamma)
        u_next = u_decayed - spikes * A_eff
        a_next = self.rho * a_prev + spikes
        return spikes, (u_next, a_next), A_eff

    def clamp_parameters(self, v_th_min: float = 0.1):
        with torch.no_grad():
            self.beta.clamp_(0.01, 0.999)
            self.v_th.clamp_(min=v_th_min)
            self.beta_ad.clamp_(min=0.0, max=5.0)

# STDP & INTRINSIC PLASTICITY

class HomeostaticIntrinsicPlasticity:
    def __init__(self, target_hz_min: float = 10.0, target_hz_max: float = 35.0, eta_ip: float = 0.005):
        self.target_hz_min = float(target_hz_min)
        self.target_hz_max = float(target_hz_max)
        self.eta_ip = float(eta_ip)

    @torch.no_grad()
    def update(self, neuron, spikes: Tensor, window_duration_sec: float):
        batch_size, n_steps, size = spikes.shape
        rate_hz = spikes.sum(dim=(0, 1)) / (batch_size * window_duration_sec)
        delta_v = torch.zeros_like(neuron.v_th)
        over = rate_hz > self.target_hz_max
        under = rate_hz < self.target_hz_min
        delta_v[over] = self.eta_ip * (rate_hz[over] - self.target_hz_max)
        delta_v[under] = -self.eta_ip * (self.target_hz_min - rate_hz[under])
        neuron.v_th.add_(delta_v).clamp_(min=0.1, max=3.0)

class STDPLayer(nn.Module):
    def __init__(self, in_features: int, out_features: int, dt_ms: float = 4.0, a_plus: float = 0.01, a_minus: float = 0.012, lr: float = 0.005):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.a_plus = a_plus
        self.a_minus = a_minus
        self.lr = lr
        self.alpha_pre = math.exp(-dt_ms / 20.0)
        self.alpha_post = math.exp(-dt_ms / 20.0)
        self.weight = nn.Parameter(torch.empty(out_features, in_features).uniform_(0.1, 0.4), requires_grad=False)
        self.post_lif = LIFNeuron(out_features, beta_init=0.85, v_th_init=1.0)
        self.normalize_weights()

    @torch.no_grad()
    def normalize_weights(self):
        self.weight.clamp_(0.0, 1.0)
        row_sums = self.weight.sum(dim=1, keepdim=True).clamp_(min=1e-6)
        self.weight.mul_(max(1.0, 0.25 * self.in_features) / row_sums)

    def init_traces(self, batch_size: int, device: torch.device):
        return (torch.zeros(batch_size, self.in_features, device=device),
                torch.zeros(batch_size, self.out_features, device=device),
                self.post_lif.init_state(batch_size, device))

    def step(self, s_pre: Tensor, traces, learn: bool = False):
        x_pre, x_post, u_post = traces
        x_pre_next = self.alpha_pre * x_pre + s_pre
        current = torch.matmul(s_pre, self.weight.t())
        s_post, u_post_next = self.post_lif.step(current, u_post)
        x_post_next = self.alpha_post * x_post + s_post
        if learn:
            with torch.no_grad():
                dw_ltp = torch.bmm(s_post.unsqueeze(2), x_pre_next.unsqueeze(1)).sum(dim=0)
                dw_ltd = torch.bmm(x_post_next.unsqueeze(2), s_pre.unsqueeze(1)).sum(dim=0)
                dW = (self.a_plus * dw_ltp - self.a_minus * dw_ltd) / float(s_pre.shape[0])
                self.weight.add_(self.lr * dW)
                self.normalize_weights()
        return s_post, (x_pre_next, x_post_next, u_post_next)

# GLOBAL SNN BACKBONE
class GlobalSNNBackbone(nn.Module):
    def __init__(self, n_input_channels: int = 38, reservoir_size: int = 128, ablation_mode: str = "full_engine", n_classes: int = 2):
        super().__init__()
        self.n_input_channels = n_input_channels
        self.reservoir_size = reservoir_size
        self.n_classes = n_classes
        self.dt_ms = 4.0

        # Ablation mode flags
        self.use_alif = "alif" in ablation_mode or ablation_mode == "full_engine"
        self.use_stdp = "stdp" in ablation_mode or ablation_mode == "full_engine"
        self.use_ip = ablation_mode == "full_engine"

        # Input projection
        self.input_proj = nn.Linear(n_input_channels, reservoir_size)
        # Recurrent weights
        self.w_rec = nn.Parameter(torch.empty(reservoir_size, reservoir_size).normal_(0.0, 1.0 / math.sqrt(reservoir_size)))
        # Reservoir neuron
        if self.use_alif:
            self.reservoir_neuron = ALIFNeuron(reservoir_size, beta_init=0.9, v_th_init=1.0, tau_ad_ms=500.0, dt_ms=self.dt_ms)
        else:
            self.reservoir_neuron = LIFNeuron(reservoir_size, beta_init=0.9, v_th_init=1.0)

        # Intrinsic Plasticity & STDP
        self.ip = HomeostaticIntrinsicPlasticity() if self.use_ip else None
        self.stdp_layer = STDPLayer(reservoir_size, reservoir_size) if self.use_stdp else None

        # Feature Readout
        self.feature_proj = nn.Linear(reservoir_size, 256)

        # Classification Readout Head (SuperSpike BPTT)
        self.classifier = nn.Linear(256, n_classes)
        self.readout_lif = LIFNeuron(n_classes, beta_init=0.9, v_th_init=1.0)

    def _simulate_reservoir(self, spikes: Tensor):
        batch, n_steps, _ = spikes.shape
        device = spikes.device
        res_state = self.reservoir_neuron.init_state(batch, device)
        s_prev = torch.zeros(batch, self.reservoir_size, device=device)
        stdp_traces = self.stdp_layer.init_traces(batch, device) if self.stdp_layer is not None else None
        feedforward = self.input_proj(spikes)
        res_spikes_list, stdp_spikes_list = [], []
        total_synops = 0

        for t in range(n_steps):
            i_in = feedforward[:, t, :]
            i_rec = torch.matmul(s_prev, self.w_rec.t())
            total_synops += int(spikes[:, t, :].sum().item() * self.reservoir_size + s_prev.sum().item() * self.reservoir_size)

            if self.use_alif:
                s_res, res_state, _ = self.reservoir_neuron.step(i_in + i_rec, res_state)
            else:
                s_res, res_state = self.reservoir_neuron.step(i_in + i_rec, res_state)

            s_prev = s_res
            res_spikes_list.append(s_res)

            if self.stdp_layer is not None:
                s_stdp, stdp_traces = self.stdp_layer.step(s_res, stdp_traces, learn=False)
                stdp_spikes_list.append(s_stdp)

        res_spikes = torch.stack(res_spikes_list, dim=1)
        if self.stdp_layer is not None:
            feature_in = 0.5 * res_spikes + 0.5 * torch.stack(stdp_spikes_list, dim=1)
        else:
            feature_in = res_spikes

        features = self.feature_proj(feature_in)  # [B, T, 256]
        sparsity = 1.0 - (res_spikes.sum().item() / max(1, res_spikes.numel()))
        return features, {"res_spikes": res_spikes, "total_synops": total_synops, "sparsity": sparsity}

    def extract(self, spikes: Tensor) -> Tensor:
        features, _ = self._simulate_reservoir(spikes)
        return features

    def forward_classify(self, spikes: Tensor):
        features, aux = self._simulate_reservoir(spikes)
        batch, n_steps, _ = features.shape
        device = features.device
        readout_u = self.readout_lif.init_state(batch, device)
        class_curr = self.classifier(features)
        out_spikes, out_mems = [], []

        for t in range(n_steps):
            s_out, readout_u = self.readout_lif.step(class_curr[:, t, :], readout_u)
            out_spikes.append(s_out)
            out_mems.append(readout_u)

        spk_tensor = torch.stack(out_spikes, dim=1)
        counts = spk_tensor.sum(dim=1)
        max_mems = torch.stack(out_mems, dim=1).amax(dim=1)
        is_top = counts == counts.amax(dim=1, keepdim=True)
        preds = max_mems.masked_fill(~is_top, float("-inf")).argmax(dim=1)
        norm_counts = counts + 0.1 * max_mems.clamp(min=0.0)
        confidence_pct = torch.softmax(norm_counts, dim=-1).amax(dim=-1) * 100.0
        aux["confidence_pct"] = confidence_pct
        return norm_counts, preds, aux

    def constrain_parameters(self):
        self.reservoir_neuron.clamp_parameters(v_th_min=0.1)
        self.readout_lif.clamp_parameters(v_th_min=0.1)
        if self.stdp_layer is not None:
            self.stdp_layer.normalize_weights()

    @torch.no_grad()
    def apply_intrinsic_plasticity(self, res_spikes: Tensor, window_duration_sec: float):
        if self.ip is not None:
            self.ip.update(self.reservoir_neuron, res_spikes, window_duration_sec)

    @torch.no_grad()
    def fit_stdp(self, spikes: Tensor):
        if self.stdp_layer is None:
            return
        batch, n_steps, _ = spikes.shape
        device = spikes.device
        res_state = self.reservoir_neuron.init_state(batch, device)
        s_prev = torch.zeros(batch, self.reservoir_size, device=device)
        feedforward = self.input_proj(spikes)
        traces = self.stdp_layer.init_traces(batch, device)
        for t in range(n_steps):
            i_in = feedforward[:, t, :]
            i_rec = torch.matmul(s_prev, self.w_rec.t())
            if self.use_alif:
                s_res, res_state, _ = self.reservoir_neuron.step(i_in + i_rec, res_state)
            else:
                s_res, res_state = self.reservoir_neuron.step(i_in + i_rec, res_state)
            s_prev = s_res
            _, traces = self.stdp_layer.step(s_res, traces, learn=True)
print("All classes defined successfully!")

Training Device: cpu
All classes defined successfully!


### Preprocess & Encode MUMTAZ Dataset


In [ ]:
CACHE_FILE = OUTPUT_DIR / "mumtaz_encoded_EC.pt"

if CACHE_FILE.exists():
    print(f"Loading pre-encoded dataset from cache: {CACHE_FILE}")
    cached_data = torch.load(CACHE_FILE, weights_only=False)
    subjects = cached_data["subjects"]
    labels = cached_data["labels"]
    eval_spikes = cached_data["eval_spikes"]
    all_spikes = cached_data["all_spikes"]
else:
    print("Encoding MUMTAZ dataset from scratch...")
    ec_files = sorted(MUMTAZ_DIR.glob("*EC.edf"))
    subjects, labels = [], {}
    eval_spikes, all_spikes = {}, {}

    for p in ec_files:
        stem = p.stem
        m_mdd = re.search(r"MDD\s+S(\d+)", stem, re.I)
        m_hc = re.search(r"(?:^|\D)H\s+S(\d+)", stem, re.I)
        if m_mdd:
            sid = f"MDD_S{int(m_mdd.group(1)):02d}"
            lbl = 1
        elif m_hc:
            sid = f"HC_S{int(m_hc.group(1)):02d}"
            lbl = 0
        else:
            continue

        if sid in labels:
            continue

        # Load EDF with MNE
        raw = mne.io.read_raw_edf(str(p), preload=True, verbose=False)
        pick_names = [ch for ch in raw.ch_names if any(std in ch for std in MUMTAZ_CHANNELS) and not "A2" in ch and not "23A" in ch and not "24A" in ch]
        if len(pick_names) != 19:
            continue
        raw.pick(pick_names)
        data = raw.get_data().astype(np.float64)
        fs = float(raw.info["sfreq"])

        # Bandpass 0.5-45 Hz
        filtered = bandpass_filter(data, fs=fs, low=0.5, high=45.0)
        # Windowing 2s (512 samples at 256 Hz)
        windows = make_windows(filtered, fs=fs, window_sec=2.0)
        if len(windows) < 10:
            continue

        # 20% Calibration Z-Score split
        n_cal = max(1, int(round(len(windows) * 0.2)))
        cal_win, eval_win = windows[:n_cal], windows[n_cal:]
        mean = cal_win.mean(axis=(0, 2), keepdims=True)
        std = cal_win.std(axis=(0, 2), keepdims=True) + 1e-8
        cal_scaled = (cal_win - mean) / std
        eval_scaled = (eval_win - mean) / std

        # Adaptive TBR thresholds (k=1.0)
        diffs = cal_scaled[:, :, 1:] - cal_scaled[:, :, :-1]
        thresholds = diffs.mean(axis=(0, 2)) + 1.0 * diffs.std(axis=(0, 2))
        thresholds = np.maximum(thresholds, 1e-6)

        # Dual-binary encoding [N, T, 38]
        cal_spk = tbr_encode_dual_binary(cal_scaled, thresholds)
        eval_spk = tbr_encode_dual_binary(eval_scaled, thresholds)

        subjects.append(sid)
        labels[sid] = lbl
        eval_spikes[sid] = eval_spk
        all_spikes[sid] = torch.cat((cal_spk, eval_spk), dim=0)

    subjects.sort()
    cached_data = {"subjects": subjects, "labels": labels, "eval_spikes": eval_spikes, "all_spikes": all_spikes}
    torch.save(cached_data, CACHE_FILE)
    print(f"Successfully encoded {len(subjects)} subjects and saved to {CACHE_FILE}")

mdd_count = sum(1 for s in subjects if labels[s] == 1)
hc_count = sum(1 for s in subjects if labels[s] == 0)
print(f"Loaded {len(subjects)} subjects: {mdd_count} MDD, {hc_count} Healthy Controls")

Loading pre-encoded dataset from cache: /content/drive/MyDrive/J26-DS-348/outputs/c2/mumtaz_encoded_EC.pt
Loaded 58 subjects: 30 MDD, 28 Healthy Controls


In [ ]:
LOCAL_CACHE_FILE = Path("/content/mumtaz_encoded_EC.pt")
print(f"Copying/Saving cache to local disk: {LOCAL_CACHE_FILE}")
torch.save(cached_data, LOCAL_CACHE_FILE)

# Re-load from local cache to verify speed and existence
cached_data = torch.load(LOCAL_CACHE_FILE, weights_only=False)
subjects = cached_data["subjects"]
labels = cached_data["labels"]
eval_spikes = cached_data["eval_spikes"]
all_spikes = cached_data["all_spikes"]
print("Local cache loaded successfully!")

Copying/Saving cache to local disk: /content/mumtaz_encoded_EC.pt
Local cache loaded successfully!


In [ ]:
import random

# Configuration Hyperparameters
SEED = 42
RESERVOIR_SIZE = 128
LR = 0.001
EPOCHS = 10
BATCH_SIZE = 16

# Set random seeds for reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Hyperparameters defined and seeds configured successfully!")

In [ ]:
benchmark_results = {}
best_models_per_mode = {}

# Configure folds
sub_labels = [labels[s] for s in subjects]
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = list(skf.split(subjects, sub_labels))

In [ ]:
# reservoir_alif
mode = "reservoir_alif"
print(f"\nStarting Training: {mode}")
fold_metrics = []
best_mode_f1 = -1.0
best_mode_model = None

for f_idx, (train_idx, test_idx) in enumerate(folds, 1):
    print(f"\n Fold {f_idx}/5 ")
    train_subs = [subjects[i] for i in train_idx]
    test_subs = [subjects[i] for i in test_idx]

    x_train = torch.cat([all_spikes[s] for s in train_subs], dim=0).to(DEVICE)
    y_train = torch.tensor([labels[s] for s in train_subs for _ in range(len(all_spikes[s]))], dtype=torch.long, device=DEVICE)
    x_test = torch.cat([eval_spikes[s] for s in test_subs], dim=0).to(DEVICE)
    y_test = torch.tensor([labels[s] for s in test_subs for _ in range(len(eval_spikes[s]))], dtype=torch.long, device=DEVICE)

    backbone = GlobalSNNBackbone(n_input_channels=38, reservoir_size=RESERVOIR_SIZE, ablation_mode=mode, n_classes=2).to(DEVICE)
    optimizer = torch.optim.Adam(backbone.parameters(), lr=LR, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    n_samples = len(x_train)

    for epoch in range(1, EPOCHS + 1):
        backbone.train()
        perm = torch.randperm(n_samples, device=DEVICE)
        epoch_loss, batches = 0.0, 0
        for b in range(0, n_samples, BATCH_SIZE):
            batch_idx = perm[b : b + BATCH_SIZE]
            bx, by = x_train[batch_idx], y_train[batch_idx]
            optimizer.zero_grad()
            logits, _, aux = backbone.forward_classify(bx)
            loss = criterion(logits, by)
            loss.backward()
            optimizer.step()
            backbone.constrain_parameters()
            epoch_loss += loss.item()
            batches += 1

        # Fast val
        backbone.eval()
        preds_list = []
        with torch.no_grad():
            for b in range(0, len(x_test), BATCH_SIZE):
                bx = x_test[b : b + BATCH_SIZE]
                _, preds, _ = backbone.forward_classify(bx)
                preds_list.append(preds.cpu())
        y_pred = torch.cat(preds_list).numpy()
        y_true = y_test.cpu().numpy()
        epoch_acc = accuracy_score(y_true, y_pred)
        epoch_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
        print(f"Epoch {epoch}/{EPOCHS} | Loss: {epoch_loss/batches:.4f} | Val Acc: {epoch_acc*100:.2f}% | Val F1: {epoch_f1:.3f}")

    backbone.eval()
    preds_list, total_synops, total_sparsity = [], 0, 0.0
    with torch.no_grad():
        for b in range(0, len(x_test), BATCH_SIZE):
            bx = x_test[b : b + BATCH_SIZE]
            _, preds, aux = backbone.forward_classify(bx)
            preds_list.append(preds.cpu())
            total_synops += aux.get("total_synops", 0)
            total_sparsity += aux.get("sparsity", 0.0) * len(bx)

    y_pred = torch.cat(preds_list).numpy()
    y_true = y_test.cpu().numpy()
    acc = accuracy_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred, average="binary", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    synops_win = total_synops / max(1, len(x_test))
    sparsity = total_sparsity / max(1, len(x_test))

    fold_metrics.append({"acc": acc, "rec": rec, "f1": f1, "synops": synops_win, "sparsity": sparsity, "cm": confusion_matrix(y_true, y_pred).tolist()})
    print(f"--> Fold {f_idx} Final: Acc: {acc*100:.2f}%, F1: {f1:.3f}")
    if f1 > best_mode_f1:
        best_mode_f1 = f1
        best_mode_model = backbone

benchmark_results[mode] = fold_metrics
best_models_per_mode[mode] = best_mode_model

# Save the best weights for this option
ckpt_path = f"/content/global_backbone_mumtaz_{mode}_best.pt"
torch.save(best_mode_model.state_dict(), ckpt_path)
print(f"Saved best weights to {ckpt_path}")

# Plot instant summary
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].bar(range(1, 6), [f["acc"]*100 for f in fold_metrics], color="teal")
ax[0].set_title("Fold Accuracies (%)")
ax[1].bar(range(1, 6), [f["f1"] for f in fold_metrics], color="orange")
ax[1].set_title("Fold Macro-F1 Scores")
ax[2].bar(range(1, 6), [f["sparsity"]*100 for f in fold_metrics], color="purple")
ax[2].set_title("Reservoir Sparsity (%)")
plt.suptitle(f"Performance Metrics - {mode}")
plt.show()